In [2]:
# Dependencies 

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
from datetime import datetime
import os
import glob
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.types import LongType
import requests

Matplotlib created a temporary cache directory at /scratch/ekim18/job_48716776/matplotlib-875m8fpa because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [3]:
# SparkSession Configuration

# 16 cores, 128GB total memory — local[*] mode
# In local mode there are no separate executor processes — all task execution
# runs as threads within a single JVM. Executor config parameters have no effect
# and are omitted. Driver memory is set to 120GB to give the JVM nearly the
# full node allocation.
spark = SparkSession.builder \
    .appName("PushshiftRedditPreprocessing") \
    .config("spark.driver.memory", "120g") \
    .config("spark.driver.maxResultSize", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Spark version: 3.5.0
Spark UI: http://exp-1-02.expanse.sdsc.edu:4040


In [4]:
# Data Load 

DATA_DIR = "../data/raw/"
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.parquet")))

COLS = ["author", "created_utc", "id", "num_comments", "score",
        "selftext", "subreddit", "subreddit_id", "title"]

pre_cutoff = [f for f in files if os.path.basename(f) >= "RS_2015-01"]
post_cutoff = [f for f in files if os.path.basename(f) < "RS_2015-01"]

# Files where created_utc is STRING - need to cast
df_pre = spark.read.parquet(*pre_cutoff) \
    .select(*[F.col(c) for c in COLS if c != 'created_utc'],
            F.col('created_utc').cast('long').alias('created_utc'))

df_post = spark.read.parquet(*post_cutoff) \
    .select(*[F.col(c) for c in COLS if c != 'created_utc'],
            F.col('created_utc').cast('long').alias('created_utc'))

MIN_TS = 1119398400  # June 2005
MAX_TS = 1700000000  # Nov 2023

df = df_pre.union(df_post) \
    .filter(F.col('created_utc').between(MIN_TS, MAX_TS))

print(f"Files loaded: {len(files)}")
print(f"Partitions: {df.rdd.getNumPartitions()}")

Files loaded: 218
Partitions: 683


In [6]:
# SparkUI Screenshot

# Get the active Spark Context and URL
sc = spark.sparkContext
url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"

# Fetch the executor data from the API
response = requests.get(url)
executors = response.json()

# Format into a readable DataFrame
sparkUI_df = pd.DataFrame(executors)[['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']]
sparkUI_df['maxMemory_GB'] = (sparkUI_df['maxMemory'] / (1024**3)).round(2)
print(sparkUI_df)

# Spark context master check
print(spark.sparkContext.master)

       id  totalCores    maxMemory  activeTasks  isActive  maxMemory_GB
0  driver          16  77120667648            0      True         71.82
local[*]


In [6]:
# Hardcode row_count to dataset size for count verifications

row_count = 549662955

In [7]:
# ── FILTERING & CLEANING ─────────────────────────────────────────────────────

df_clean = df \
    .filter(F.col("subreddit").isNotNull()) \
    .filter(F.col("subreddit_id").isNotNull()) \
    .filter(F.col("score").isNotNull()) \
    .filter(F.col("num_comments") >= 0) \
    .dropDuplicates(["id"])

print(f"Original row count:      {row_count:,}")
clean_count = df_clean.count()
print(f"Row count after filters: {clean_count:,}")
print(f"Rows removed:            {row_count - clean_count:,}")

Original row count:      549,662,955
Row count after filters: 549,355,364
Rows removed:            307,591


## Bot Author Handling

Rather than attempting frequency-based bot detection (which would require a full groupBy on author + date across 549M rows to compute per-author daily post rates), we take a simpler and more interpretable approach: flagging known high-volume bot accounts identified in EDA as a binary feature `is_known_bot`.

This preserves all bot-authored posts in the dataset — bot posts have real and consistent engagement patterns (clustering heavily in low-engagement) that the model can learn from. Filtering them out would discard genuine signal. The `is_known_bot` flag gives the model explicit information about author type without requiring expensive per-author temporal aggregations.

The known bot list is seeded from the top authors output in EDA. Frequency-based detection can be revisited in as feature engineering improvement.

In [8]:
# ── BOT AUTHOR FLAGGING ───────────────────────────────────────────────────────

KNOWN_BOTS = {
    "AutoModerator", "AutoNewsAdmin", "AutoNewspaperAdmin",
    "politicbot", "RPBot", "ImagesOfNetwork", "-en-",
    "KellyfromLeedsUK"
}

df_clean = df_clean \
    .withColumn("is_known_bot",
        F.when(F.col("author").isin(list(KNOWN_BOTS)), 1)
         .otherwise(0)) \
    .withColumn("is_anonymous_author",
        F.when(
            (F.col("author") == "[deleted]") | (F.col("author") == ""), 1)
         .otherwise(0)) \
    .withColumn("has_title",
        F.when(F.length(F.col("title")) == 0, 0)
         .otherwise(1))

print("Author type features added.")
df_clean.select("is_known_bot", "is_anonymous_author", "has_title").show(5)

Author type features added.


Py4JJavaError: An error occurred while calling o143.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 351 in stage 10.0 failed 1 times, most recent failure: Lost task 351.0 in stage 10.0 (TID 1322) (exp-1-02.expanse.sdsc.edu executor driver): java.io.IOException: No space left on device
	at java.base/java.io.FileOutputStream.writeBytes(Native Method)
	at java.base/java.io.FileOutputStream.write(FileOutputStream.java:349)
	at org.apache.spark.storage.TimeTrackingOutputStream.write(TimeTrackingOutputStream.java:59)
	at org.apache.spark.io.MutableCheckedOutputStream.write(MutableCheckedOutputStream.scala:43)
	at java.base/java.io.BufferedOutputStream.flushBuffer(BufferedOutputStream.java:81)
	at java.base/java.io.BufferedOutputStream.write(BufferedOutputStream.java:127)
	at net.jpountz.lz4.LZ4BlockOutputStream.flushBufferedData(LZ4BlockOutputStream.java:225)
	at net.jpountz.lz4.LZ4BlockOutputStream.write(LZ4BlockOutputStream.java:178)
	at java.base/java